# Aula 9 — Word Embeddings: palavras como vetores densos

**Da contagem de termos à representação distribuída**

Até aqui, representamos texto com vocabulários explícitos: cada termo ocupava sua própria dimensão em Bag-of-Words, TF-IDF e n-grams.

Agora faremos uma mudança importante de paradigma:

> em vez de representar uma palavra por uma coluna exclusiva, vamos representá-la por um vetor denso de números.

Essa ideia está no coração dos **word embeddings**.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar o que é um embedding;
- distinguir vetores esparsos e vetores densos;
- compreender a ideia de representação distribuída;
- interpretar similaridade por cosseno;
- entender, em alto nível, Word2Vec e FastText;
- treinar um pequeno Word2Vec com `gensim`;
- inspecionar palavras mais próximas no espaço vetorial;
- reconhecer limitações de embeddings estáticos.


## 📘 Glossário da aula

Conceitos centrais: **embedding · vetor denso · representação distribuída · similaridade por cosseno · Word2Vec · FastText · embedding estático**.

- [Glossário PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)
- [Glossary EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.


## 2. Do vetor esparso ao vetor denso

Em Bag-of-Words, uma palavra pode ser representada indiretamente por uma dimensão exclusiva em um espaço muito grande.

Exemplo conceitual:

```text
atendimento → [0, 0, 1, 0, 0, 0, ...]
suporte     → [0, 0, 0, 0, 1, 0, ...]
```

Em embeddings, cada palavra passa a ocupar um vetor denso:

```text
atendimento → [ 0.18, -0.42, 0.77, ... ]
suporte     → [ 0.21, -0.39, 0.71, ... ]
```

A proximidade entre vetores pode capturar relações de uso aprendidas a partir do corpus.


## 3. Representação distribuída

Nos embeddings, o significado não fica concentrado em uma única posição do vetor.

Ele é distribuído entre várias dimensões aprendidas.

Por isso, duas palavras usadas em contextos semelhantes podem acabar próximas no espaço vetorial.

Essa ideia está ligada à hipótese distribucional:

> palavras que aparecem em contextos semelhantes tendem a ter usos semelhantes.


## 4. Similaridade por cosseno

Uma forma comum de comparar embeddings é a **similaridade por cosseno**.

Ela mede o ângulo entre dois vetores, priorizando direção em vez de magnitude absoluta.

Valores próximos de `1` indicam vetores com direções muito parecidas.


In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
b = np.array([1.1, 1.9, 3.2])

cosine_similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print("Similaridade por cosseno:", round(float(cosine_similarity), 3))


### O que observar

Os vetores são diferentes, mas apontam em direções semelhantes.

**Checkpoint 1:** similaridade alta não significa identidade; significa proximidade geométrica segundo essa medida.


## 5. Word2Vec em alto nível

Word2Vec aprende embeddings a partir dos contextos das palavras.

Duas arquiteturas clássicas são:

- **CBOW**: tenta prever uma palavra a partir de seu contexto;
- **Skip-gram**: tenta prever palavras de contexto a partir de uma palavra central.

Nesta aula, o objetivo não é derivar matematicamente o algoritmo, mas compreender o mecanismo conceitual e observar seu comportamento.


## 6. Um corpus minúsculo para experimentar

Vamos criar frases com padrões simples de atendimento, elogio e reclamação.


In [ ]:
sentences = [
    ["atendimento", "excelente", "rápido"],
    ["suporte", "excelente", "eficiente"],
    ["atendimento", "rápido", "eficiente"],
    ["suporte", "rápido", "prestativo"],
    ["atendimento", "ruim", "demorado"],
    ["suporte", "ruim", "lento"],
    ["cliente", "satisfeito", "atendimento"],
    ["cliente", "satisfeito", "suporte"],
    ["cliente", "insatisfeito", "demorado"],
    ["cliente", "insatisfeito", "lento"],
]

sentences[:3]


## 7. Treinando um Word2Vec pequeno

Usaremos `gensim.models.Word2Vec`.

Como o corpus é minúsculo, o resultado deve ser interpretado apenas como demonstração didática.


In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=sentences,
    vector_size=20,
    window=2,
    min_count=1,
    workers=1,
    sg=1,
    epochs=200,
    seed=42,
)

print("Dimensão do embedding:", model.wv.vector_size)


### Observando um vetor

A palavra `atendimento` agora possui uma representação numérica densa.


In [ ]:
vector = model.wv["atendimento"]
print(vector)
print("Número de dimensões:", len(vector))


## 8. Palavras próximas no espaço vetorial

Agora podemos perguntar quais palavras ficaram mais próximas de `atendimento` segundo a similaridade por cosseno.


In [ ]:
model.wv.most_similar("atendimento", topn=5)


### Interpretação cuidadosa

Se `suporte` aparecer próximo de `atendimento`, isso não significa que o modelo compreendeu essas palavras como uma pessoa compreenderia.

Significa que, neste corpus, elas apareceram em padrões de contexto semelhantes.

**Checkpoint 2:** embeddings aprendem regularidades do corpus — inclusive seus vieses, limitações e ruídos.


## 9. Word2Vec versus TF-IDF

Uma comparação útil:

```text
TF-IDF
→ uma dimensão por termo
→ vetor geralmente esparso
→ ótima interpretabilidade lexical

Word embedding
→ poucas dezenas/centenas de dimensões
→ vetor denso
→ relações distribuídas aprendidas do contexto
```

Nenhuma representação é universalmente melhor. Elas respondem a necessidades diferentes.


## 10. E o FastText?

FastText amplia a ideia de Word2Vec ao representar palavras também por partes menores, como subpalavras.

Isso pode ajudar em idiomas com muitas variações morfológicas e em palavras raras ou não vistas exatamente durante o treinamento.

Exemplo conceitual:

```text
atendimento
→ pedaços/subpalavras
→ representação combinada
```


## 11. Limitação dos embeddings estáticos

Em Word2Vec e FastText clássicos, uma palavra possui essencialmente o mesmo embedding independentemente da frase.

Mas considere:

```text
banco aprovou o crédito
sentei no banco da praça
```

A palavra `banco` tem sentidos diferentes, mas um embedding estático não cria automaticamente uma representação diferente para cada ocorrência.

Essa limitação será importante quando avançarmos para embeddings contextuais e transformers.


## 12. Exercício guiado

Use o modelo treinado nesta aula.

Seu código deve:

1. obter o vetor de `suporte`;
2. exibir sua dimensionalidade;
3. calcular a similaridade entre `suporte` e `atendimento` usando `model.wv.similarity()`;
4. listar as três palavras mais próximas de `suporte`.


In [ ]:
# Escreva sua solução aqui.

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q9.hint()` e `q9.solution()`.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q9 = TILExercise(
    hint_text=(
        "Use a interface `model.wv` do **gensim**. "
        "Acesse um vetor com `model.wv['suporte']`, compare palavras com `model.wv.similarity(...)` "
        "e obtenha vizinhos com `model.wv.most_similar(..., topn=3)`."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "support_vector = model.wv['suporte']\n"
        "print('Dimensionalidade:', len(support_vector))\n"
        "print('Similaridade:', round(model.wv.similarity('suporte', 'atendimento'), 3))\n"
        "print(model.wv.most_similar('suporte', topn=3))\n"
        "```"
    ),
)

print("Exercício preparado. Tente resolver antes de usar q9.hint() ou q9.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q9.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q9.solution()


## 13. Reprodutibilidade

- linguagem: Python;
- bibliotecas: `numpy` e `gensim`;
- algoritmo: Word2Vec;
- arquitetura demonstrativa: Skip-gram (`sg=1`);
- semente: `42`;
- `workers=1` para reduzir variação de execução;
- acelerador: CPU;
- internet: desabilitada;
- corpus externo: nenhum.


## 14. Resumo

Nesta aula, você aprendeu que:

- embeddings representam palavras por vetores densos;
- significado distribucional emerge de padrões de contexto;
- similaridade por cosseno mede proximidade angular;
- Word2Vec aprende relações a partir de contextos;
- FastText incorpora informação de subpalavras;
- embeddings estáticos têm limitações para palavras polissêmicas;
- representações densas abrem caminho para técnicas contextuais mais avançadas.

### Ideia principal

```text
Em vez de perguntar apenas se uma palavra apareceu,
passamos a perguntar onde ela está no espaço de representações.
```

**Fim da Aula 9.**
